# Using the `tommos-adapter` in Ubermag

This notebook shows how to use the `tommos-adapter` for Ubermag.

In [1]:
import shlex
import subprocess

import micromagneticmodel as mm
import numpy as np
import pyvista as pv

import tommos_adapter as ta

## Generate mesh

We use `tommos` to generate the mesh:

In [2]:
cmd = "tommos mesh --geom box --extent 20,20,20 --h 2 --out-name cube"
subprocess.run(shlex.split(cmd), cwd="input_files")

[ok] Wrote data: cube.npz (nodes=3284, tets=15109)
[ok] Wrote visualization: cube.vtu (cell_data: mat_id)


CompletedProcess(args=['tommos', 'mesh', '--geom', 'box', '--extent', '20,20,20', '--h', '2', '--out-name', 'cube'], returncode=0)

## Initialize System

We start with a `System` object, with name `hysteresis`.

In [3]:
system = mm.System(name="hysteresis")

## Define magnetization

We assume the mesh is already divided in zones (with names `id`):

In [4]:
state = pv.read("input_files/cube.vtu")
id_array = state.get_array("mat_id")
state.cell_data["id"] = id_array  # Rename `mat_id` cell data to `id`
state.cell_data.remove("mat_id")  # remove old data with name `mat_id`
state

UnstructuredGrid (0x7f87eb437700)
  N Cells:    15109
  N Points:   3284
  X Bounds:   -1.000e+01, 1.000e+01
  Y Bounds:   -1.000e+01, 1.000e+01
  Z Bounds:   -1.000e+01, 1.000e+01
  N Arrays:   1

We define an initial magnetization, constant (0, 0, 1)

In [5]:
m = np.zeros((state.n_points, 3), dtype=np.float32)
m[:, 2] = 1
m

array([[0., 0., 1.],
       [0., 0., 1.],
       [0., 0., 1.],
       ...,
       [0., 0., 1.],
       [0., 0., 1.],
       [0., 0., 1.]], shape=(3284, 3), dtype=float32)

We add the magnetization vector to the state object as vector field

In [6]:
state.point_data["m"] = m
state

UnstructuredGrid (0x7f87eb437700)
  N Cells:    15109
  N Points:   3284
  X Bounds:   -1.000e+01, 1.000e+01
  Y Bounds:   -1.000e+01, 1.000e+01
  Z Bounds:   -1.000e+01, 1.000e+01
  N Arrays:   2

Add Spontaneous Polarization property as cell constant

In [7]:
Js = 1.76  # spontaneous polarization in Tesla
Js_array = Js * np.ones_like(state.n_cells, dtype=np.float32)
state.cell_data["Js"] = Js_array
state

UnstructuredGrid (0x7f87eb437700)
  N Cells:    15109
  N Points:   3284
  X Bounds:   -1.000e+01, 1.000e+01
  Y Bounds:   -1.000e+01, 1.000e+01
  Z Bounds:   -1.000e+01, 1.000e+01
  N Arrays:   3

In [8]:
system.m = state

## Define energy terms

Energy terms include intrinsic properties

In [9]:
A = 1e-12  # exchange stiffness constant in J/m
K1 = 4e5  # first uniaxial anisotropy constant in J/m^3
u = (0, 0, 1)  # anisotropy unit axis
system.energy = mm.Exchange(A=A) + mm.UniaxialAnisotropy(K=K1, u=u)
system.energy

Exchange(A=1e-12) + UniaxialAnisotropy(K=400000.0, u=(0, 0, 1))

## Define driver and hysteresis parameters

Next, we define the driver. The initialization does not need any argument.

In [10]:
hd = ta.HysteresisDriver()
hd

HysteresisDriver()

Let us simulate hysteresis between $-1$ Tesla and $1$ Tesla applied along the $z$-direction in steps of $0.2$ Tesla:

In [11]:
Hmin = (0, 0, -1 / mm.consts.mu0)
Hmax = (0, 0, 1 / mm.consts.mu0)
n = 21

Now we execute the hysteresis loop.

In [12]:
hd.drive(system, Hmin=Hmin, Hmax=Hmax, n=n)

AttributeError: 'HysteresisDriver' object has no attribute 'krn_path'

In [27]:
e = system.energy[0]
if isinstance(e, mm.Exchange):
    print(e.A)

1e-12


In [43]:
type(system.energy[1])

micromagneticmodel.energy.uniaxialanisotropy.UniaxialAnisotropy

In [45]:
system.energy[1].u

(0, 0, 1)